In [1]:
import os
import sklearn
import requests
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.preprocessing import StandardScaler
import itertools
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

/Users/olamideoluwalade/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
with open("home_data.json", "r") as data:
    home_data = json.load(data) 

In [3]:
# Sample user liked houses
liked_houses_id = ['6786270504', '6541999527', "5332041691"]
user_preferences = ["pet friendly", "garage", "barbecue"]

In [4]:
def jaccard_similarity(loc1, loc2):
# Calculates Jaccard similarity between neighborhoods.
    if not loc1 or not loc2 or not loc1.get("neighborhoods") or not loc2.get("neighborhoods"):  # Handle cases where location or neighborhood data is missing.
        return 0

    neighborhoods1 = {n.get("name") for n in loc1.get("neighborhoods", [])}  # Extract neighborhood names for the first house.
    neighborhoods2 = {n.get("name") for n in loc2.get("neighborhoods", [])}  # Extract neighborhood names for the second house.

    intersection = neighborhoods1.intersection(neighborhoods2)  # Find the common neighborhoods.
    union = neighborhoods1.union(neighborhoods2)  # Find the union of all neighborhoods.
    # print(union)

    if not union:  #Handle empty sets
        return 0

    return len(intersection) / len(union)  # Calculate and return the Jaccard similarity.

In [6]:

def compare_range(val1, val2, weight = 1):
    # Compares two values within a range and returns a score.
    if val1 is None or val2 is None:  # Handle cases where either value is None.
        return 0
    if val1 == val2:  # If the values are exactly the same, return the full weight.
        return weight
    range_diff = abs(val1 - val2)  # Calculate the absolute difference between the values.
    if range_diff <= 1:  # If the difference is within 1 (you can adjust this range), return half the weight.
        return weight * 0.5  # Partial weight
    return 0  # Otherwise, return 0 (not similar enough).

In [7]:
def compare_features(liked_house, house, user_preferences):
    """Compares features of two houses and returns a similarity score."""
    score = 0  # Initialize the score for this house comparison.

    # Type matching
    if liked_house.get("description") and house.get("description"):  # Check if both houses have description data.
        if liked_house["description"].get("type") == house["description"].get("type"):  # If the house types are the same, add 1 to the score.
            score += 1

        # Beds and Baths
        score += compare_range(liked_house["description"].get("beds_min"), house["description"].get("beds_min"))  # Compare number of beds (using the compare_range function).
        score += compare_range(liked_house["description"].get("baths_min"), house["description"].get("baths_min"))  # Compare number of baths.

        # Square footage
        if liked_house["description"].get("sqft_min") and house["description"].get("sqft_min"):  # If square footage data is available for both houses.
            score += compare_range(liked_house["description"]["sqft_min"], house["description"]["sqft_min"], 0.25)  # Compare square footage (with a lower weight of 0.25).

    # Neighborhood matching (using Jaccard similarity)
    score += jaccard_similarity(liked_house.get("location"), house.get("location")) * 0.5  # Calculate Jaccard similarity between neighborhoods and add it to the score (weighted by 0.5).

    # Pet policy (exact match)
    if liked_house.get("pet_policy") and house.get("pet_policy"):  # Check if both houses have pet policy data.
        if liked_house["pet_policy"].get("cats") == house["pet_policy"].get("cats"):  # If cat policies are the same, add 0.25 to the score.
            score += 0.25  # Lower weight for pet policies
        if liked_house["pet_policy"].get("dogs") == house["pet_policy"].get("dogs"):  # If dog policies are the same, add 0.25 to the score.
            score += 0.25

    liked_price = liked_house.get("list_price")  # Assuming this exists
    house_price = house.get("list_price")
    if liked_price and house_price:
        price_diff_percentage = abs(liked_price - house_price) / liked_price if liked_price != 0 else 0  # avoid division by zero
        price_similarity = 1 - min(price_diff_percentage, 1)  # Similarity between 0 and 1
        score += price_similarity * 0.3  # Weight price similarity

    # 2. Amenity Matching (More sophisticated):
    liked_amenities = set()
    liked_details = liked_house.get("details")  # Get details, or None
    if liked_details:  # Check if details is not None
        for detail in liked_details:
            if detail.get("category") == "Amenities":
                amenities_list = detail.get("text")
                if amenities_list:
                    for amenity in amenities_list:
                        liked_amenities.add(amenity.lower())

    house_amenities = set()
    house_details = house.get("details") # Get details, or None
    if house_details: # Check if details is not None
        for detail in house_details:
            if detail.get("category") == "Amenities":
                amenities_list = detail.get("text")
                if amenities_list:
                    for amenity in amenities_list:
                        house_amenities.add(amenity.lower())

    common_amenities = liked_amenities.intersection(house_amenities)
    # print(common_amenities)
    score += len(common_amenities) * 0.1  # Weight amenity matching

    # 3. Keyword Matching in Description (TF-IDF or simple count):
    liked_description = liked_house.get("description", {}).get("text", "").lower()
    house_description = house.get("description", {}).get("text", "").lower()

    if liked_description and house_description:
        liked_keywords = set(liked_description.split())  # Very basic keyword extraction
        house_keywords = set(house_description.split())
        common_keywords = liked_keywords.intersection(house_keywords)
        score += len(common_keywords) * 0.05 #Lower weight

        # More advanced:
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform([liked_description, house_description])
        from sklearn.metrics.pairwise import cosine_similarity
        similarity = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0]
        score += similarity * 0.1 #Adjust weight

    # 4. Property Size Ratio:
    liked_sqft = liked_house.get("description", {}).get("sqft_min")
    house_sqft = house.get("description", {}).get("sqft_min")
    if liked_sqft and house_sqft:
        sqft_ratio = min(liked_sqft, house_sqft) / max(liked_sqft, house_sqft)
        score += sqft_ratio * 0.2 # Weight square footage ratio

    # 5. User Preference Matching
    house_features = set()  # Collect all relevant features from the house

    house_details_user_pref = house.get("details")
    if house_details_user_pref: # Check if it is none
        for detail in house_details_user_pref:
            if detail.get("category") == "Amenities":
                for amenity in detail.get("text", []): # Add this to avoid errors if text is None
                    house_features.add(amenity.lower())

    cats_policy = house.get("pet_policy", {}).get("cats")
    if isinstance(cats_policy, str):
        house_features.add(cats_policy.lower())

    dogs_policy = house.get("pet_policy", {}).get("dogs")
    if isinstance(dogs_policy, str):
        house_features.add(dogs_policy.lower())

    # Extract keywords from description
    description = house.get("description", {}).get("text", "").lower()
    if description:
        house_features.update(description.split())

    # Add location features
    location = house.get("location", {})
    for neighborhood in location.get("neighborhoods", []):
        house_features.add(neighborhood.get("name","").lower())

    #Add property type
    if house.get("description"):
        house_features.add(house["description"].get("type","").lower())


    for preference in user_preferences:
        if preference.lower() in house_features:
            score += 0.2  # Weight user preferences (adjust as needed)
            
    print(house_features)
    # print(score)
    return score



In [8]:
def get_relevant_items(liked_houses_id_user):
    """Returns a list of relevant item IDs for a given user (Improved)."""
    relevant_items = liked_houses_id_user #Treat liked houses as relevant items
    return relevant_items

def precision_at_k(recommended_items, relevant_items, k):
    intersection = set(recommended_items[:k]) & set(relevant_items)  # Items in top K recommendations that are also relevant
    # print(len(intersection) / k if k > 0 else 0 )
    return len(intersection) / k if k > 0 else 0  # Avoid division by zero

def recall_at_k(recommended_items, relevant_items, k):
    intersection = set(recommended_items[:k]) & set(relevant_items)
    # print(len(intersection) / len(relevant_items) if len(relevant_items) > 0 else 0)
    return len(intersection) / len(relevant_items) if len(relevant_items) > 0 else 0  # Avoid division by zero

def map_at_k(recommended_items_list, relevant_items_list, k):
    """Calculates Mean Average Precision at K."""
    average_precision_sum = 0
    for recommended_items, relevant_items in zip(recommended_items_list, relevant_items_list):
        average_precision = 0
        num_relevant = 0
        for i, item in enumerate(recommended_items[:k]):
            if item in relevant_items:
                num_relevant += 1
                average_precision += num_relevant / (i + 1)  # Precision at each rank
        average_precision = average_precision / len(relevant_items) if len(relevant_items) > 0 else 0
        average_precision_sum += average_precision
    return average_precision_sum / len(recommended_items_list) if len(recommended_items_list) > 0 else 0


In [9]:
def recommend_houses(liked_houses_id, data):
    """Recommends houses based on user's liked houses using content-based filtering."""

    if not liked_houses_id or not data or not data.get("data") or not data["data"].get("results"):
        return []  # Handle empty or invalid input: If any of the input parameters are empty or None, return an empty list (no recommendations).

    all_houses = data["data"]["results"]  # Extract the list of all house data from the input dictionary.
    liked_houses = [house for house in all_houses if house["property_id"] in liked_houses_id]  # Create a list of the actual house data for the houses the user liked.  This uses a list comprehension for conciseness.

    if not liked_houses:
        return []  # If none of the liked house IDs are found in the data, return an empty list.

    recommended_houses_all_users = []  # List to store recommendations for all users
    relevant_items_all_users = []  # List to store relevant items for all users

    recommended_ids = []

    for i, liked_houses_id_user in enumerate(liked_houses_id): #Iterate through each user if there are multiple users in the liked_houses_id
        liked_houses_user = [house for house in all_houses if house["property_id"] in liked_houses_id_user]

        recommended_houses = []
        for house in all_houses:
            if house["property_id"] not in liked_houses_id_user:
                score = 0
                for liked_house in liked_houses_user:
                    score += compare_features(liked_house, house, user_preferences)

                if score > 0:
                    recommended_houses.append((house["property_id"], score))

        recommended_houses.sort(key=lambda x: x[1], reverse=True)

        recommended_ids_user = [house_id for house_id, score in recommended_houses]
        recommended_ids.append(recommended_ids_user)

        # For evaluation (assuming you have a way to get relevant items):
        relevant_items_user = get_relevant_items(liked_houses_id_user)  # Replace with your logic
        recommended_houses_all_users.append(recommended_ids_user)
        relevant_items_all_users.append(relevant_items_user)
    k = 50
    # Calculate and print evaluation metrics
    precision_at_k_all_users = [precision_at_k(recommended_houses_all_users[i], relevant_items_all_users[i], k) for i in range(len(recommended_houses_all_users))]
    recall_at_k_all_users = [recall_at_k(recommended_houses_all_users[i], relevant_items_all_users[i], k) for i in range(len(recommended_houses_all_users))]
    map_at_k_value = map_at_k(recommended_houses_all_users, relevant_items_all_users, k)

    print(f"Average Precision@{k}: {np.mean(precision_at_k_all_users)}")
    print(f"Average Recall@{k}: {np.mean(recall_at_k_all_users)}")
    print(f"MAP@{k}: {map_at_k_value}")

    return recommended_ids #Return recommended list for all users

In [10]:
recommended_ids = recommend_houses(liked_houses_id, home_data)  # Call the recommendation function.
# print(recommended_ids)  # Print the list of recommended house IDs.
# recommended_ids
count = 0

{'northeast washington', 'apartment', 'eckington'}
{'apartment', 'u-street', 'northwest washington'}
{'apartment', 'buzzard point', 'southwest washington', 'southwest waterfront'}
{'apartment', 'northwest washington', 'dupont circle'}
{'hill east', 'capitol hill', 'apartment', 'southeast washington'}
{'apartment', 'northwest washington', 'north cleveland park'}
{'northeast washington', 'apartment', 'brookland', 'university heights'}
{'apartment', 'columbia heights', 'northwest washington'}
{'apartment', 'southeast washington', 'anacostia', 'randle highlands'}
{'apartment', 'capitol hill', 'southeast washington'}
{'apartment', 'southeast washington', 'anacostia', 'randle highlands'}
{'apartment', 'logan circle', 'northwest washington', 'logan circle historic district'}
{'apartment', 'petworth', 'northwest washington'}
{'northeast washington', 'apartment'}
{'hill east', 'capitol hill', 'apartment', 'southeast washington'}
{'apartment', 'near southeast', 'navy yard', 'southeast washington